In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import joblib
import csv
import os
from datetime import datetime


MODEL_FILE = "Customer_Support_Best_Model.pkl"
EVALUATION_FILE = "Human_Evaluation.csv"

EXPECTED_CLASSES = [
    "Complaint",
    "Question",
    "Positive",
    "Other"
]


MODEL = None
MODEL_ERROR = None

try:
    MODEL = joblib.load(MODEL_FILE)

    if not hasattr(MODEL, "predict"):
        raise ValueError(
            "The loaded file does not contain a valid "
            "scikit-learn prediction model."
        )

except Exception as exc:
    MODEL_ERROR = str(exc)


class CustomerSupportDashboard(tk.Tk):

    def __init__(self):
        super().__init__()

        self.title(
            "Customer Support Tweet Classification & Human Evaluation"
        )

        self.geometry("1200x780")
        self.minsize(950, 650)
        self.configure(bg="#F4F6F8")

        self.last_tweet = ""
        self.last_prediction = ""

        self.setup_style()
        self.create_variables()
        self.create_header()
        self.create_notebook()
        self.create_prediction_tab()
        self.create_evaluation_tab()
        self.create_status_bar()

        self.bind(
            "<Control-Return>",
            lambda event: self.predict()
        )

        self.bind(
            "<Escape>",
            lambda event: self.clear_tweet()
        )

        self.protocol(
            "WM_DELETE_WINDOW",
            self.exit_application
        )

        if MODEL is None:
            self.status_var.set("Model loading error")
            self.after(300, self.show_model_error)
        else:
            self.status_var.set(
                "Model loaded successfully — Ready"
            )

    def setup_style(self):

        style = ttk.Style(self)

        try:
            style.theme_use("clam")
        except tk.TclError:
            pass

        style.configure(
            "TNotebook",
            background="#F4F6F8",
            borderwidth=0
        )

        style.configure(
            "TNotebook.Tab",
            font=("Segoe UI", 11, "bold"),
            padding=(22, 12)
        )

        style.configure(
            "TRadiobutton",
            font=("Segoe UI", 10),
            background="white"
        )

    def create_variables(self):

        self.prediction_var = tk.StringVar(
            value="Awaiting Prediction"
        )

        self.confidence_var = tk.StringVar(
            value="Confidence: —"
        )

        self.status_var = tk.StringVar(
            value="Loading model..."
        )

        self.human_rating_var = tk.StringVar(
            value="Correct"
        )

        self.ease_var = tk.IntVar(value=5)
        self.design_var = tk.IntVar(value=5)
        self.quality_var = tk.IntVar(value=5)

        self.probability_vars = {
            label: tk.StringVar(value="0.00%")
            for label in EXPECTED_CLASSES
        }

    def create_header(self):

        header = tk.Frame(
            self,
            bg="#17202A",
            height=105
        )

        header.pack(fill="x")
        header.pack_propagate(False)

        tk.Label(
            header,
            text="Customer Support Tweet Classification",
            bg="#17202A",
            fg="white",
            font=("Segoe UI", 22, "bold")
        ).pack(
            anchor="w",
            padx=30,
            pady=(18, 0)
        )

        tk.Label(
            header,
            text="Machine Learning Prediction & Human Evaluation Dashboard",
            bg="#17202A",
            fg="#BFC9D4",
            font=("Segoe UI", 10)
        ).pack(
            anchor="w",
            padx=32,
            pady=(3, 0)
        )

    def create_notebook(self):

        container = tk.Frame(
            self,
            bg="#F4F6F8"
        )

        container.pack(
            fill="both",
            expand=True,
            padx=22,
            pady=(18, 10)
        )

        self.notebook = ttk.Notebook(container)

        self.notebook.pack(
            fill="both",
            expand=True
        )

        self.prediction_frame = tk.Frame(
            self.notebook,
            bg="#F4F6F8"
        )

        self.evaluation_frame = tk.Frame(
            self.notebook,
            bg="#F4F6F8"
        )

        self.notebook.add(
            self.prediction_frame,
            text="  Tweet Classification  "
        )

        self.notebook.add(
            self.evaluation_frame,
            text="  Human Evaluation  "
        )

    def create_prediction_tab(self):

        frame = self.prediction_frame

        frame.grid_columnconfigure(
            0,
            weight=3
        )

        frame.grid_columnconfigure(
            1,
            weight=2
        )

        frame.grid_rowconfigure(
            0,
            weight=1
        )

        input_card = tk.Frame(
            frame,
            bg="white",
            highlightbackground="#D9DEE3",
            highlightthickness=1
        )

        input_card.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 10),
            pady=5
        )

        input_card.grid_rowconfigure(
            1,
            weight=1
        )

        input_card.grid_columnconfigure(
            0,
            weight=1
        )

        tk.Label(
            input_card,
            text="Tweet Input",
            bg="white",
            fg="#17202A",
            font=("Segoe UI", 15, "bold")
        ).grid(
            row=0,
            column=0,
            sticky="w",
            padx=22,
            pady=(20, 10)
        )

        text_frame = tk.Frame(
            input_card,
            bg="white"
        )

        text_frame.grid(
            row=1,
            column=0,
            sticky="nsew",
            padx=22
        )

        text_frame.grid_rowconfigure(
            0,
            weight=1
        )

        text_frame.grid_columnconfigure(
            0,
            weight=1
        )

        self.tweet_text = tk.Text(
            text_frame,
            wrap="word",
            font=("Segoe UI", 12),
            bg="#FAFBFC",
            fg="#17202A",
            insertbackground="#17202A",
            relief="flat",
            padx=15,
            pady=15,
            undo=True
        )

        self.tweet_text.grid(
            row=0,
            column=0,
            sticky="nsew"
        )

        scrollbar = ttk.Scrollbar(
            text_frame,
            orient="vertical",
            command=self.tweet_text.yview
        )

        scrollbar.grid(
            row=0,
            column=1,
            sticky="ns"
        )

        self.tweet_text.configure(
            yscrollcommand=scrollbar.set
        )

        button_frame = tk.Frame(
            input_card,
            bg="white"
        )

        button_frame.grid(
            row=2,
            column=0,
            sticky="ew",
            padx=22,
            pady=20
        )

        for column in range(4):
            button_frame.grid_columnconfigure(
                column,
                weight=1
            )

        tk.Button(
            button_frame,
            text="Predict",
            command=self.predict,
            bg="#1F6FEB",
            fg="white",
            activebackground="#1557B0",
            activeforeground="white",
            relief="flat",
            font=("Segoe UI", 10, "bold"),
            cursor="hand2",
            padx=12,
            pady=10
        ).grid(
            row=0,
            column=0,
            sticky="ew",
            padx=(0, 5)
        )

        tk.Button(
            button_frame,
            text="Clear",
            command=self.clear_tweet,
            bg="#E9ECEF",
            fg="#17202A",
            activebackground="#D6DADF",
            relief="flat",
            font=("Segoe UI", 10, "bold"),
            cursor="hand2",
            padx=12,
            pady=10
        ).grid(
            row=0,
            column=1,
            sticky="ew",
            padx=5
        )

        tk.Button(
            button_frame,
            text="Load Example",
            command=self.load_example,
            bg="#E9ECEF",
            fg="#17202A",
            activebackground="#D6DADF",
            relief="flat",
            font=("Segoe UI", 10, "bold"),
            cursor="hand2",
            padx=12,
            pady=10
        ).grid(
            row=0,
            column=2,
            sticky="ew",
            padx=5
        )

        tk.Button(
            button_frame,
            text="Exit",
            command=self.exit_application,
            bg="#C0392B",
            fg="white",
            activebackground="#922B21",
            activeforeground="white",
            relief="flat",
            font=("Segoe UI", 10, "bold"),
            cursor="hand2",
            padx=12,
            pady=10
        ).grid(
            row=0,
            column=3,
            sticky="ew",
            padx=(5, 0)
        )

        result_card = tk.Frame(
            frame,
            bg="white",
            highlightbackground="#D9DEE3",
            highlightthickness=1
        )

        result_card.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(10, 0),
            pady=5
        )

        result_card.grid_columnconfigure(
            0,
            weight=1
        )

        tk.Label(
            result_card,
            text="Prediction Result",
            bg="white",
            fg="#17202A",
            font=("Segoe UI", 15, "bold")
        ).grid(
            row=0,
            column=0,
            sticky="w",
            padx=22,
            pady=(20, 12)
        )

        self.result_card = tk.Frame(
            result_card,
            bg="#F1F5F9",
            height=130
        )

        self.result_card.grid(
            row=1,
            column=0,
            sticky="ew",
            padx=22,
            pady=(0, 15)
        )

        self.result_card.grid_propagate(False)

        self.result_label = tk.Label(
            self.result_card,
            textvariable=self.prediction_var,
            bg="#F1F5F9",
            fg="#17202A",
            font=("Segoe UI", 20, "bold")
        )

        self.result_label.pack(
            expand=True,
            pady=(15, 0)
        )

        self.confidence_label = tk.Label(
            self.result_card,
            textvariable=self.confidence_var,
            bg="#F1F5F9",
            fg="#566573",
            font=("Segoe UI", 11)
        )

        self.confidence_label.pack(
            pady=(0, 15)
        )

        tk.Label(
            result_card,
            text="Prediction Probabilities",
            bg="white",
            fg="#17202A",
            font=("Segoe UI", 12, "bold")
        ).grid(
            row=2,
            column=0,
            sticky="w",
            padx=22,
            pady=(5, 12)
        )

        self.progress_bars = {}

        probability_colors = {
            "Complaint": "#E74C3C",
            "Question": "#3498DB",
            "Positive": "#27AE60",
            "Other": "#8E44AD"
        }

        for index, label in enumerate(EXPECTED_CLASSES):

            row = 3 + index * 2

            tk.Label(
                result_card,
                text=label,
                bg="white",
                fg="#34495E",
                font=("Segoe UI", 10, "bold")
            ).grid(
                row=row,
                column=0,
                sticky="w",
                padx=22,
                pady=(5, 0)
            )

            tk.Label(
                result_card,
                textvariable=self.probability_vars[label],
                bg="white",
                fg="#566573",
                font=("Segoe UI", 9)
            ).grid(
                row=row,
                column=0,
                sticky="e",
                padx=22,
                pady=(5, 0)
            )

            bar_frame = tk.Frame(
                result_card,
                bg="#E5E7EB",
                height=18
            )

            bar_frame.grid(
                row=row + 1,
                column=0,
                sticky="ew",
                padx=22,
                pady=(2, 5)
            )

            canvas = tk.Canvas(
                bar_frame,
                height=18,
                bg="#E5E7EB",
                highlightthickness=0
            )

            canvas.pack(
                fill="both",
                expand=True
            )

            self.progress_bars[label] = (
                canvas,
                probability_colors[label]
            )

        tk.Label(
            result_card,
            text="Tip: Press Ctrl + Enter to predict.",
            bg="white",
            fg="#7F8C8D",
            font=("Segoe UI", 9, "italic")
        ).grid(
            row=11,
            column=0,
            sticky="w",
            padx=22,
            pady=(20, 15)
        )

    def create_evaluation_tab(self):

        frame = self.evaluation_frame

        frame.grid_columnconfigure(
            0,
            weight=1
        )

        frame.grid_rowconfigure(
            1,
            weight=1
        )

        tk.Label(
            frame,
            text="Human Evaluation",
            bg="#F4F6F8",
            fg="#17202A",
            font=("Segoe UI", 18, "bold")
        ).grid(
            row=0,
            column=0,
            sticky="w",
            padx=10,
            pady=(5, 15)
        )

        card = tk.Frame(
            frame,
            bg="white",
            highlightbackground="#D9DEE3",
            highlightthickness=1
        )

        card.grid(
            row=1,
            column=0,
            sticky="nsew",
            padx=5,
            pady=5
        )

        card.grid_columnconfigure(
            1,
            weight=1
        )

        tk.Label(
            card,
            text="Tweet",
            bg="white",
            font=("Segoe UI", 10, "bold")
        ).grid(
            row=0,
            column=0,
            sticky="nw",
            padx=20,
            pady=(20, 8)
        )

        self.eval_tweet = tk.Text(
            card,
            height=4,
            wrap="word",
            font=("Segoe UI", 10),
            bg="#FAFBFC",
            relief="solid",
            borderwidth=1
        )

        self.eval_tweet.grid(
            row=0,
            column=1,
            sticky="ew",
            padx=(0, 20),
            pady=(20, 8)
        )

        tk.Label(
            card,
            text="Model Prediction",
            bg="white",
            font=("Segoe UI", 10, "bold")
        ).grid(
            row=1,
            column=0,
            sticky="w",
            padx=20,
            pady=8
        )

        self.eval_prediction_label = tk.Label(
            card,
            text="—",
            bg="#F1F5F9",
            fg="#17202A",
            font=("Segoe UI", 11, "bold"),
            anchor="w",
            padx=10,
            pady=8
        )

        self.eval_prediction_label.grid(
            row=1,
            column=1,
            sticky="ew",
            padx=(0, 20),
            pady=8
        )

        tk.Label(
            card,
            text="Human Rating",
            bg="white",
            font=("Segoe UI", 10, "bold")
        ).grid(
            row=2,
            column=0,
            sticky="w",
            padx=20,
            pady=8
        )

        rating_frame = tk.Frame(
            card,
            bg="white"
        )

        rating_frame.grid(
            row=2,
            column=1,
            sticky="w",
            padx=(0, 20),
            pady=8
        )

        ttk.Radiobutton(
            rating_frame,
            text="Correct",
            variable=self.human_rating_var,
            value="Correct"
        ).pack(
            side="left",
            padx=(0, 25)
        )

        ttk.Radiobutton(
            rating_frame,
            text="Incorrect",
            variable=self.human_rating_var,
            value="Incorrect"
        ).pack(
            side="left"
        )

        self.create_rating_row(
            card,
            3,
            "Ease of Use",
            self.ease_var
        )

        self.create_rating_row(
            card,
            4,
            "Interface Design",
            self.design_var
        )

        self.create_rating_row(
            card,
            5,
            "Prediction Quality",
            self.quality_var
        )

        tk.Label(
            card,
            text="Comments",
            bg="white",
            font=("Segoe UI", 10, "bold")
        ).grid(
            row=6,
            column=0,
            sticky="nw",
            padx=20,
            pady=(12, 8)
        )

        self.comments_text = tk.Text(
            card,
            height=6,
            wrap="word",
            font=("Segoe UI", 10),
            bg="#FAFBFC",
            relief="solid",
            borderwidth=1
        )

        self.comments_text.grid(
            row=6,
            column=1,
            sticky="ew",
            padx=(0, 20),
            pady=(12, 8)
        )

        tk.Button(
            card,
            text="Save Evaluation",
            command=self.save_evaluation,
            bg="#27AE60",
            fg="white",
            activebackground="#1E8449",
            activeforeground="white",
            relief="flat",
            font=("Segoe UI", 11, "bold"),
            cursor="hand2",
            padx=20,
            pady=10
        ).grid(
            row=7,
            column=1,
            sticky="e",
            padx=(0, 20),
            pady=15
        )

    def create_rating_row(
        self,
        parent,
        row,
        label,
        variable
    ):

        tk.Label(
            parent,
            text=label,
            bg="white",
            font=("Segoe UI", 10, "bold")
        ).grid(
            row=row,
            column=0,
            sticky="w",
            padx=20,
            pady=8
        )

        rating_frame = tk.Frame(
            parent,
            bg="white"
        )

        rating_frame.grid(
            row=row,
            column=1,
            sticky="w",
            padx=(0, 20),
            pady=8
        )

        for value in range(1, 6):

            ttk.Radiobutton(
                rating_frame,
                text=str(value),
                variable=variable,
                value=value
            ).pack(
                side="left",
                padx=(0, 15)
            )

    def create_status_bar(self):

        status = tk.Frame(
            self,
            bg="#17202A",
            height=32
        )

        status.pack(fill="x")
        status.pack_propagate(False)

        tk.Label(
            status,
            textvariable=self.status_var,
            bg="#17202A",
            fg="#D5DBDB",
            font=("Segoe UI", 9),
            anchor="w"
        ).pack(
            side="left",
            padx=15
        )

        tk.Label(
            status,
            text="Model: Customer_Support_Best_Model.pkl",
            bg="#17202A",
            fg="#AAB7B8",
            font=("Segoe UI", 9)
        ).pack(
            side="right",
            padx=15
        )

    def show_model_error(self):

        if MODEL_ERROR:

            messagebox.showerror(
                "Model Loading Error",
                "Customer_Support_Best_Model.pkl could not be loaded.\n\n"
                f"Error:\n{MODEL_ERROR}\n\n"
                "Make sure the model file was saved with joblib "
                "and is in the same folder as this Python file."
            )

    def predict(self):

        user_text = (
            self.tweet_text
            .get("1.0", "end-1c")
            .strip()
        )

        if not user_text:

            messagebox.showwarning(
                "Input Required",
                "Please enter a tweet before clicking Predict."
            )

            return

        if MODEL is None:

            messagebox.showerror(
                "Model Error",
                f"The model could not be loaded.\n\n{MODEL_ERROR}"
            )

            return

        try:

            prediction = MODEL.predict(
                [user_text]
            )[0]

            probabilities_raw = MODEL.predict_proba(
                [user_text]
            )[0]

            classes = MODEL.classes_

            probabilities = {}

            for class_name, probability in zip(
                classes,
                probabilities_raw
            ):

                probabilities[str(class_name)] = float(
                    probability
                )

            prediction_label = str(
                prediction
            )

            confidence = probabilities.get(
                prediction_label,
                max(probabilities.values())
            )

            self.last_tweet = user_text
            self.last_prediction = prediction_label

            self.prediction_var.set(
                prediction_label
            )

            self.confidence_var.set(
                f"Confidence: {confidence * 100:.2f}%"
            )

            for label in EXPECTED_CLASSES:

                value = probabilities.get(
                    label,
                    0.0
                )

                self.probability_vars[
                    label
                ].set(
                    f"{value * 100:.2f}%"
                )

                self.update_probability_bar(
                    label,
                    value
                )

            self.update_result_card(
                prediction_label
            )

            self.eval_tweet.delete(
                "1.0",
                "end"
            )

            self.eval_tweet.insert(
                "1.0",
                user_text
            )

            self.eval_prediction_label.config(
                text=prediction_label
            )

            self.status_var.set(
                f"Prediction completed: {prediction_label}"
            )

        except Exception as exc:

            messagebox.showerror(
                "Prediction Error",
                f"{exc}"
            )

            self.status_var.set(
                "Prediction failed"
            )

    def update_probability_bar(
        self,
        label,
        value
    ):

        canvas, bar_color = self.progress_bars[label]

        canvas.delete("all")

        width = canvas.winfo_width()

        if width <= 1:
            width = 300

        bar_width = int(
            width * max(
                0,
                min(value, 1)
            )
        )

        canvas.create_rectangle(
            0,
            0,
            bar_width,
            18,
            fill=bar_color,
            outline=""
        )

    def update_result_card(
        self,
        prediction
    ):

        card_colors = {

            "Complaint": (
                "#FDEDEC",
                "#C0392B"
            ),

            "Question": (
                "#EBF5FB",
                "#2874A6"
            ),

            "Positive": (
                "#EAFAF1",
                "#239B56"
            ),

            "Other": (
                "#F5EEF8",
                "#7D3C98"
            )
        }

        background, foreground = card_colors.get(
            prediction,
            (
                "#F1F5F9",
                "#17202A"
            )
        )

        self.result_card.config(
            bg=background
        )

        self.result_label.config(
            bg=background,
            fg=foreground
        )

        self.confidence_label.config(
            bg=background,
            fg=foreground
        )

    def clear_tweet(self):

        self.tweet_text.delete(
            "1.0",
            "end"
        )

        self.prediction_var.set(
            "Awaiting Prediction"
        )

        self.confidence_var.set(
            "Confidence: —"
        )

        for label in EXPECTED_CLASSES:

            self.probability_vars[
                label
            ].set(
                "0.00%"
            )

            canvas, _ = self.progress_bars[label]

            canvas.delete("all")

        self.result_card.config(
            bg="#F1F5F9"
        )

        self.result_label.config(
            bg="#F1F5F9",
            fg="#17202A"
        )

        self.confidence_label.config(
            bg="#F1F5F9",
            fg="#566573"
        )

        self.last_tweet = ""
        self.last_prediction = ""

        self.status_var.set(
            "Input cleared"
        )

    def load_example(self):

        example = (
            "I am really happy with the support team. "
            "They solved my problem quickly and professionally!"
        )

        self.tweet_text.delete(
            "1.0",
            "end"
        )

        self.tweet_text.insert(
            "1.0",
            example
        )

        self.status_var.set(
            "Example tweet loaded"
        )

    def save_evaluation(self):

        tweet = (
            self.eval_tweet
            .get("1.0", "end-1c")
            .strip()
        )

        model_prediction = (
            self.eval_prediction_label
            .cget("text")
        )

        human_rating = (
            self.human_rating_var
            .get()
        )

        ease_of_use = (
            self.ease_var
            .get()
        )

        interface_design = (
            self.design_var
            .get()
        )

        prediction_quality = (
            self.quality_var
            .get()
        )

        comments = (
            self.comments_text
            .get("1.0", "end-1c")
            .strip()
        )

        if not tweet:

            messagebox.showwarning(
                "Missing Tweet",
                "Please enter or predict a tweet "
                "before saving the evaluation."
            )

            return

        if model_prediction == "—":

            messagebox.showwarning(
                "Missing Prediction",
                "Please run the model prediction "
                "before saving the evaluation."
            )

            return

        file_exists = os.path.exists(
            EVALUATION_FILE
        )

        headers = [
            "Timestamp",
            "Tweet",
            "Model Prediction",
            "Human Rating",
            "Ease of Use",
            "Interface Design",
            "Prediction Quality",
            "Comments"
        ]

        row = [
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            tweet,
            model_prediction,
            human_rating,
            ease_of_use,
            interface_design,
            prediction_quality,
            comments
        ]

        try:

            with open(
                EVALUATION_FILE,
                "a",
                newline="",
                encoding="utf-8-sig"
            ) as csv_file:

                writer = csv.writer(
                    csv_file
                )

                if not file_exists:
                    writer.writerow(
                        headers
                    )

                writer.writerow(
                    row
                )

            messagebox.showinfo(
                "Evaluation Saved",
                "Human evaluation saved successfully.\n\n"
                f"File: {EVALUATION_FILE}"
            )

            self.comments_text.delete(
                "1.0",
                "end"
            )

            self.status_var.set(
                f"Evaluation saved to {EVALUATION_FILE}"
            )

        except Exception as exc:

            messagebox.showerror(
                "Save Error",
                f"Could not save the evaluation:\n\n{exc}"
            )

    def exit_application(self):

        answer = messagebox.askyesno(
            "Exit Application",
            "Are you sure you want to exit?"
        )

        if answer:
            self.destroy()


if __name__ == "__main__":

    app = CustomerSupportDashboard()

    app.mainloop()